# Luxembourg And Timor-Leste Case Studies With `scalable_distances`

This notebook recreates the two paper cases requested for the promoted package in `Research-Sandbox/scalable_general_distances_per_country`.

It is self-contained in two senses. First, the markdown states the case definitions, parameters, and reported paper results. Second, the executable cells demonstrate the same logic with the new API on tiny in-memory networks, so the notebook can run quickly on a fresh development machine without downloading country-scale data. The final section shows the real-data CLI/API calls to run the cases against the shared cache.

## Case Definitions From The Paper

| Case | Purpose | Paper pipeline settings | Paper-scale reported result |
| --- | --- | --- | --- |
| Luxembourg schools | Descriptive school access from generated road-network matrix | `amenity=school`, no population aggregation, retained matrix threshold `1,000 m`, context map saved | 394 OSM school features, 102,061 population targets representing 478,010 people; 266,225 people within 1,000 m of an OSM school |
| Timor-Leste health | Prescriptive maximum covering from generated road-network matrix | default health amenities, aggregate factor `8`, retained matrix threshold `10 km`, generated candidate grid spacing `5 km` | 28,162 population targets, 860 sources, 100,997 retained distance rows; with `p=20` and 10 km coverage, Gurobi and HiGHS both cover 757,053 represented people |

The notebook below mirrors the data contracts used by those cases: source and target tables, router strategy, matrix output modes, and optimization-ready sparse coverage neighborhoods.

In [ ]:
from itertools import combinations
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "scalable_distances").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_ROOT = PROJECT_ROOT / "src"
assert (SRC_ROOT / "scalable_distances").exists(), f"Could not find scalable_distances under {PROJECT_ROOT}"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from scalable_distances import describe_backends, describe_country_sources, write_distance_matrix
from scalable_distances.config import CountryDataSources
from scalable_distances.pipeline import ProductionRunConfig
from scalable_distances.routing.base import NetworkData
from scalable_distances.routing.strategies import NetworkXRouter

## Backend Detection

The new package treats routing engines as strategies. NetworkX is portable and does not import Pandana. Pandana is imported only when `PandanaRouter` is selected, which matters for deployments where Pandana is unavailable because of NumPy binary compatibility.

In [ ]:
describe_backends()

## Source Resolution For The Two Countries

`describe_country_sources()` resolves Geofabrik and WorldPop filenames, URLs, and cache paths without downloading. This is the first reproducibility check for both cases.

In [ ]:
shared_cache = Path(r"C:\local\Download_Depot")

luxembourg_sources = describe_country_sources(
    country_slug="luxembourg",
    iso3="LUX",
    base_dir=shared_cache / "luxembourg_data",
    worldpop_dataset="global1",
    worldpop_year=2020,
    worldpop_filename="lux_ppp_2020.tif",
)

timor_leste_sources = describe_country_sources(
    country_slug="east-timor",
    iso3="TLS",
    base_dir=shared_cache / "east-timor_data",
    worldpop_dataset="global1",
    worldpop_year=2020,
    worldpop_filename="tls_ppp_2020.tif",
)

pd.DataFrame([luxembourg_sources, timor_leste_sources])[[
    "country_slug", "iso3", "pbf_filename", "worldpop_filename", "base_dir"
]]

## Case 1: Luxembourg Descriptive School Access

Paper case: extract OSM schools as existing source locations, convert native WorldPop cells to target locations, compute road-network distances, then report how much represented population is within 500 m and 1,000 m of at least one school. The descriptive model is:

\[
\text{covered}_i(R)=1 \quad \text{if}\quad \min_{j \in J_{schools}} d_{ij}\le R.
\]

A target contributes only once at each threshold, even when several schools are within reach. For school ranking, each covered target is assigned to its nearest school before population is summed.

In [ ]:
lux_network = NetworkData(
    nodes=pd.DataFrame([
        {"node_id": 1, "lon": 6.10, "lat": 49.60},
        {"node_id": 2, "lon": 6.11, "lat": 49.61},
        {"node_id": 3, "lon": 6.12, "lat": 49.62},
        {"node_id": 4, "lon": 6.13, "lat": 49.63},
    ]),
    edges=pd.DataFrame([
        {"u": 1, "v": 2, "length_m": 420.0}, {"u": 2, "v": 1, "length_m": 420.0},
        {"u": 2, "v": 3, "length_m": 390.0}, {"u": 3, "v": 2, "length_m": 390.0},
        {"u": 3, "v": 4, "length_m": 480.0}, {"u": 4, "v": 3, "length_m": 480.0},
    ]),
)

lux_schools = pd.DataFrame([
    {"source_id": "school_centre", "source_type": "amenities", "lon": 6.10, "lat": 49.60},
    {"source_id": "school_north", "source_type": "amenities", "lon": 6.13, "lat": 49.63},
])
lux_population = pd.DataFrame([
    {"target_id": "pop_1", "target_type": "population", "lon": 6.10, "lat": 49.60, "population": 80.0},
    {"target_id": "pop_2", "target_type": "population", "lon": 6.12, "lat": 49.62, "population": 125.0},
    {"target_id": "pop_3", "target_type": "population", "lon": 6.13, "lat": 49.63, "population": 65.0},
])

lux_router = NetworkXRouter()
lux_router.prepare(lux_network, {})
lux_matrix = lux_router.route_many(
    lux_router.snap(lux_schools),
    lux_router.snap(lux_population),
)
lux_matrix

In [ ]:
def descriptive_coverage(matrix, targets, thresholds):
    rows = []
    for threshold in thresholds:
        covered_ids = set(matrix.loc[matrix["total_dist"] <= threshold, "target_id"])
        covered = targets[targets["target_id"].isin(covered_ids)]
        rows.append({
            "threshold_m": threshold,
            "covered_population": covered["population"].sum(),
            "covered_points": len(covered),
            "covered_pct": covered["population"].sum() / targets["population"].sum(),
        })
    return pd.DataFrame(rows)

lux_coverage = descriptive_coverage(lux_matrix, lux_population, thresholds=[500, 1000])
lux_coverage

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
plot_data = lux_coverage.assign(covered_pct=lambda df: df["covered_pct"] * 100)
ax.bar(plot_data["threshold_m"].astype(str), plot_data["covered_population"], color="#2f6f73")
for _, row in plot_data.iterrows():
    ax.text(str(int(row["threshold_m"])), row["covered_population"] + 4, f"{row['covered_pct']:.1f}%", ha="center")
ax.set_xlabel("Walking threshold (m)")
ax.set_ylabel("Covered represented population")
ax.set_title("Luxembourg school access coverage demo")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()


In [ ]:
lux_outputs = write_distance_matrix(
    lux_matrix,
    output_dir=PROJECT_ROOT / "diagnostics" / "luxembourg_case",
    run_tag="luxembourg_schools_demo",
    mode="both",
)
{key: str(path) for key, path in lux_outputs.paths.items()}

## Case 2: Timor-Leste Maximum Covering

Paper case: use the default health amenity list plus generated 5 km candidate sites. Population is aggregated with factor 8, the retained matrix threshold is 10 km, and the optimization chooses `p=20` source locations. The exact maximum-covering model is:

\[
\max \sum_i w_i z_i, \quad z_i \le \sum_{j \in N(i)} y_j, \quad \sum_j y_j \le p, \quad z_i,y_j\in\{0,1\}.
\]

The small example below uses the same sparse-neighborhood construction and solves it by enumeration so the notebook remains dependency-light. The country-scale run should use the shared maximum-covering solver from the optimization codebase.

In [ ]:
timor_network = NetworkData(
    nodes=pd.DataFrame([
        {"node_id": 1, "lon": 125.55, "lat": -8.56},
        {"node_id": 2, "lon": 125.57, "lat": -8.57},
        {"node_id": 3, "lon": 125.59, "lat": -8.58},
        {"node_id": 4, "lon": 125.61, "lat": -8.59},
    ]),
    edges=pd.DataFrame([
        {"u": 1, "v": 2, "length_m": 3000.0}, {"u": 2, "v": 1, "length_m": 3000.0},
        {"u": 2, "v": 3, "length_m": 4200.0}, {"u": 3, "v": 2, "length_m": 4200.0},
        {"u": 3, "v": 4, "length_m": 2600.0}, {"u": 4, "v": 3, "length_m": 2600.0},
    ]),
)
timor_sources = pd.DataFrame([
    {"source_id": "clinic_existing", "source_type": "amenities", "lon": 125.55, "lat": -8.56},
    {"source_id": "candidate_west", "source_type": "candidates", "lon": 125.57, "lat": -8.57},
    {"source_id": "candidate_east", "source_type": "candidates", "lon": 125.61, "lat": -8.59},
])
timor_population = pd.DataFrame([
    {"target_id": "village_1", "target_type": "population", "lon": 125.55, "lat": -8.56, "population": 180.0},
    {"target_id": "village_2", "target_type": "population", "lon": 125.59, "lat": -8.58, "population": 260.0},
    {"target_id": "village_3", "target_type": "population", "lon": 125.61, "lat": -8.59, "population": 210.0},
])

timor_router = NetworkXRouter()
timor_router.prepare(timor_network, {})
timor_matrix = timor_router.route_many(
    timor_router.snap(timor_sources),
    timor_router.snap(timor_population),
)
timor_matrix

In [ ]:
def brute_force_max_cover(matrix, targets, p, radius_m):
    source_ids = sorted(matrix["source_id"].unique())
    population = targets.set_index("target_id")["population"].to_dict()
    neighborhoods = {
        source_id: set(matrix.loc[(matrix["source_id"] == source_id) & (matrix["total_dist"] <= radius_m), "target_id"])
        for source_id in source_ids
    }
    best = None
    for selected in combinations(source_ids, p):
        covered_targets = set().union(*(neighborhoods[source_id] for source_id in selected))
        covered_population = sum(population[target_id] for target_id in covered_targets)
        candidate = {"selected": selected, "covered_targets": sorted(covered_targets), "covered_population": covered_population}
        if best is None or candidate["covered_population"] > best["covered_population"]:
            best = candidate
    return best

timor_solution = brute_force_max_cover(timor_matrix, timor_population, p=2, radius_m=10_000)
timor_solution

In [ ]:
selected_sources = set(timor_solution["selected"])
covered_targets = set(timor_solution["covered_targets"])

fig, ax = plt.subplots(figsize=(6, 4))
for _, row in timor_network.edges.iloc[::2].iterrows():
    a = timor_network.nodes.set_index("node_id").loc[row["u"]]
    b = timor_network.nodes.set_index("node_id").loc[row["v"]]
    ax.plot([a["lon"], b["lon"]], [a["lat"], b["lat"]], color="#999999", linewidth=2, zorder=1)

ax.scatter(timor_population["lon"], timor_population["lat"], s=timor_population["population"], c="#f0b429", label="Population", zorder=2)
ax.scatter(timor_sources["lon"], timor_sources["lat"], s=90, c=["#1b6ca8" if sid in selected_sources else "#9b9b9b" for sid in timor_sources["source_id"]], marker="^", label="Sources", zorder=3)
for _, row in timor_sources.iterrows():
    ax.text(row["lon"], row["lat"] + 0.002, row["source_id"], fontsize=8, ha="center")
for _, row in timor_population.iterrows():
    label = row["target_id"] + (" covered" if row["target_id"] in covered_targets else "")
    ax.text(row["lon"], row["lat"] - 0.003, label, fontsize=8, ha="center")
ax.set_title(f"Timor-Leste p=2 maximum-cover demo: {timor_solution['covered_population']:.0f} people covered")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()


In [ ]:
timor_outputs = write_distance_matrix(
    timor_matrix,
    output_dir=PROJECT_ROOT / "diagnostics" / "timor_leste_case",
    run_tag="timor_leste_health_demo",
    mode="both",
)
{key: str(path) for key, path in timor_outputs.paths.items()}

## Real-Data Runs Through The New Runner

The cells above are tiny, executable reconstructions. To run the actual country-scale cases against the shared cache, use the production runner. Luxembourg can be run directly as an amenities-to-population matrix. Timor-Leste in the paper includes generated 5 km candidate sites; the current runner handles OSM amenities end-to-end and keeps candidate generation as an extension point, so the exact paper-scale candidate experiment should be run once candidate generation is enabled or by using a candidate table as an added source layer.

Luxembourg CLI:

```powershell
py -m scalable_distances.cli run `
  --country-slug luxembourg `
  --iso3 LUX `
  --base-dir C:\\local\\Download_Depot\\luxembourg_data `
  --output-dir C:\\local\\Download_Depot\\luxembourg_data\\outputs `
  --run-tag luxembourg_schools_networkx `
  --amenity school `
  --router networkx `
  --matrix-output-mode both
```

Timor-Leste health amenities CLI scaffold:

```powershell
py -m scalable_distances.cli run `
  --country-slug east-timor `
  --iso3 TLS `
  --base-dir C:\\local\\Download_Depot\\east-timor_data `
  --output-dir C:\\local\\Download_Depot\\east-timor_data\\outputs `
  --run-tag timor_leste_health_networkx `
  --amenity hospital --amenity clinic --amenity doctors --amenity dentist --amenity pharmacy --amenity health_post --amenity nursing_home --amenity social_facility `
  --router networkx `
  --aggregate-factor 8 `
  --matrix-output-mode both
```

Use `--router pandana` only in an environment where Pandana is installed and compatible with the active NumPy version.

In [ ]:
luxembourg_production_config = ProductionRunConfig(
    sources=CountryDataSources(
        country_slug="luxembourg",
        iso3="LUX",
        base_dir=shared_cache / "luxembourg_data",
        worldpop_filename="lux_ppp_2020.tif",
    ),
    output_dir=shared_cache / "luxembourg_data" / "outputs",
    run_tag="luxembourg_schools_networkx",
    amenity_values=("school",),
    router="networkx",
    matrix_output_mode="both",
)

timor_leste_production_config = ProductionRunConfig(
    sources=CountryDataSources(
        country_slug="east-timor",
        iso3="TLS",
        base_dir=shared_cache / "east-timor_data",
        worldpop_filename="tls_ppp_2020.tif",
    ),
    output_dir=shared_cache / "east-timor_data" / "outputs",
    run_tag="timor_leste_health_networkx",
    amenity_values=("hospital", "clinic", "doctors", "dentist", "pharmacy", "health_post", "nursing_home", "social_facility"),
    router="networkx",
    aggregate_factor=8,
    matrix_output_mode="both",
)

luxembourg_production_config, timor_leste_production_config